In [1]:
import joblib

import numpy as np
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer ,PorterStemmer

from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer

from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB  
from hmmlearn.hmm import GaussianHMM

# from sktime.detection.hmm_learn import GaussianHMM 

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.decomposition import TruncatedSVD #instead of PCA
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler


In [2]:
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\20100\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\20100\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\20100\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\20100\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
stop_words=set(stopwords.words('english'))

# Load the data + class Weights

In [4]:
data = joblib.load('news_data.pkl') # original
x_train = data['X_train']
y_train = data['y_train']
x_test = data['X_test']
y_test = data['y_test']

data_balance = joblib.load('news_data_resampled.pkl') # oversample only
x_train_ran_res = data_balance['X_train']
y_train_ran_res = data_balance['y_train']
x_test_ran_res = data_balance['X_test']
y_test_ran_res = data_balance['y_test']

data_balance_rosrus=joblib.load('news_data_bal_ros_rus.pkl')  #rosrus
x_train_bal = data_balance_rosrus['X_train']
y_train_bal = data_balance_rosrus['y_train']
x_test_bal = data_balance_rosrus['X_test']
y_test_bal = data_balance_rosrus['y_test']

In [ ]:
data_undersampled=joblib.load('news_data_undersampled.pkl') # undersample only
x_train_undersampled=data_undersampled['X_train']
y_train_undersampled=data_undersampled['y_train']
x_test_undersampled=data_undersampled['X_test']
y_test_undersampled=data_undersampled['y_test']

In [12]:
class_weights_dict=joblib.load('classWeightsDic')

# Initial  POS

In [5]:
def pos_features(text, remove_stopwords=False):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    tokens = word_tokenize(text.lower())

    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words]

    pos_tags = nltk.pos_tag(tokens)
    pos_tokens = [tag for _, tag in pos_tags]
    return pos_tokens


In [6]:
num_classes = len(set(y_train))  
num_classes

10

# Original Data

In [9]:
len(x_train)

167616

## BoW feature Extraction + scalling

## count vectorizer

In [10]:
vectorizer_POS_ = CountVectorizer(tokenizer=lambda x:pos_features(x, remove_stopwords=True))
x_train_POS = vectorizer_POS_.fit_transform(x_train)
x_test_POS= vectorizer_POS_.transform(x_test)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [11]:
print(x_train_POS)

  (0, 11)	3
  (0, 14)	2
  (0, 27)	1
  (0, 7)	1
  (1, 11)	2
  (1, 14)	1
  (1, 30)	1
  (1, 6)	1
  (1, 29)	2
  (1, 18)	1
  (2, 11)	5
  (2, 7)	2
  (2, 30)	1
  (3, 11)	2
  (3, 7)	2
  (3, 26)	1
  (4, 11)	2
  (4, 14)	3
  (4, 7)	2
  (4, 29)	1
  (4, 26)	1
  (5, 11)	5
  (5, 27)	1
  (5, 7)	1
  (5, 30)	1
  :	:
  (167610, 14)	1
  (167610, 7)	1
  (167610, 29)	1
  (167610, 26)	1
  (167611, 11)	5
  (167611, 14)	1
  (167611, 27)	1
  (167611, 7)	1
  (167611, 29)	1
  (167612, 11)	2
  (167612, 14)	1
  (167612, 7)	2
  (167613, 11)	3
  (167613, 14)	1
  (167613, 7)	2
  (167613, 29)	1
  (167613, 18)	1
  (167613, 26)	1
  (167614, 11)	3
  (167614, 7)	2
  (167614, 18)	1
  (167615, 11)	6
  (167615, 14)	1
  (167615, 7)	1
  (167615, 29)	1


In [12]:
pos_counts = np.asarray(x_train_POS.sum(axis=0)).ravel()
pos_names = vectorizer_POS_.get_feature_names_out()

pos_freq = pd.Series(pos_counts, index=pos_names).sort_values(ascending=False)
pos_freq

NN      463065
NNS     179788
JJ      178187
VBP      66311
VBG      43699
VBZ      34250
RB       33917
VBD      28797
VB       20555
VBN      12995
IN       11031
JJS       6877
CD        6495
MD        6150
PRP       4135
JJR       2808
RBR       1731
DT        1728
RP        1010
FW         878
CC         656
NNP        650
RBS        446
WRB        225
WP         114
WP$         97
NNPS        79
UH          71
WDT         57
PRP$        49
TO          39
POS          9
''           8
SYM          5
EX           5
dtype: int64

In [13]:
scaler = StandardScaler(with_mean=False)
x_train_pos_scaled = scaler.fit_transform(x_train_POS)
x_test_pos_scaled  = scaler.transform(x_test_POS)

In [14]:
vectorizer_POS_stopKept = CountVectorizer(tokenizer=lambda x: pos_features(x, remove_stopwords=False))
x_train_POS_stopkept = vectorizer_POS_stopKept.fit_transform(x_train)
x_test_POS_stopkept = vectorizer_POS_stopKept.transform(x_test)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [15]:
scaler_stopkept = StandardScaler(with_mean=False)
x_train_pos_stopkept_scaled = scaler_stopkept.fit_transform(x_train_POS_stopkept)
x_test_pos_stopkept_scaled  = scaler_stopkept.transform(x_test_POS_stopkept)

## TF-IDF vectorizer

In [16]:
vectorizer_POS_tfidf = TfidfVectorizer(tokenizer=lambda x: pos_features(x, remove_stopwords=True))
x_train_POS_tfidf = vectorizer_POS_tfidf.fit_transform(x_train)
x_test_POS_tfidf = vectorizer_POS_tfidf.transform(x_test)

In [17]:
scaler_tfidf = StandardScaler(with_mean=False)
x_train_pos_tfidf_scaled = scaler_tfidf.fit_transform(x_train_POS_tfidf)
x_test_pos_tfidf_scaled  = scaler_tfidf.transform(x_test_POS_tfidf)

In [7]:
vectorizer_POS_stopkept_tfidf = TfidfVectorizer(tokenizer=lambda x: pos_features(x, remove_stopwords=False))
x_train_POS_stopkept_tfidf = vectorizer_POS_stopkept_tfidf.fit_transform(x_train)
x_test_POS_stopkept_tfidf = vectorizer_POS_stopkept_tfidf.transform(x_test)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [8]:
scaler_stopkept_tfidf = StandardScaler(with_mean=False)
x_train_pos_stopkept_tfidf_scaled = scaler_stopkept_tfidf.fit_transform(x_train_POS_stopkept_tfidf)
x_test_pos_stopkept_tfidf_scaled  = scaler_stopkept_tfidf.transform(x_test_POS_stopkept_tfidf)

## Models

In [ ]:
# results_POS_original={}

In [9]:
results_POS_original=joblib.load('results_POS_original')
results_POS_original

{'SVC_BoW StopWords removed': 0.13430378236487292,
 'MultinomialNB_BoW StopWords removed': 0.40746927574275144,
 'MLP StopWords removed': 0.4539076482519986,
 'SVC_POS StopWords kept': 0.16735473093902875,
 'MultinomialNB_POS StopWords kept': 0.37441832716859563,
 'MLP StopWords kept': 0.45825080539315116,
 '(tf-idf) SVC_POS StopWords removed': 0.12208566996778428,
 '(tf-idf) MultinomialNB_POS StopWords removed': 0.39661138288986997,
 '(tf-idf) MLP StopWords removed': 0.45426560076363204,
 '(tf-idf) SVC_POS StopWords kept': 0.12208566996778428,
 '(tf-idf) MultinomialNB_POS StopWords kept': 0.39661138288986997,
 '(tf-idf) MLP StopWords kept': 0.45426560076363204,
 'HMM StopWords removed': 0.03667820069204152,
 'HMM StopWords kept': 0.0530962892256294,
 '(tf-idf) HMM StopWords removed': 0.04724973153561628,
 '(tf-idf) HMM StopWords kept': 0.08564610428349839}

In [22]:
x_train_pos_tfidf_scaled.shape

(167616, 35)

In [23]:
x_train_pos_scaled.shape

(167616, 35)

In [13]:
models_POS={
    "SVC_POS": SVC(class_weight=class_weights_dict),
    "MultinomialNB_POS": MultinomialNB(),
    "MLP" :MLPClassifier(hidden_layer_sizes=(32,16),activation='relu',solver='adam',max_iter=100,alpha=0.001,early_stopping=True,random_state=42),
    # 'HMM' :GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
    }

## with countvectorizer

### stop words removed

In [25]:
for model_name, model in models_POS.items():
    model.fit(x_train_pos_scaled, y_train)

    y_pred_train = model.predict(x_train_pos_scaled)
    accuracy_train=accuracy_score(y_train, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_pos_scaled)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name}Testing: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)
    results_POS_original[model_name+" StopWords removed"] = accuracy

Results for SVC_BoW Training: accuracy=0.14853593928980527
Results for SVC_BoWTesting: accuracy=0.13430378236487292
              precision    recall  f1-score   support

           1       0.24      0.17      0.20      7120
           2       0.20      0.34      0.25      3589
           3       0.15      0.26      0.19      3473
           4       0.12      0.24      0.16      1980
           5       0.11      0.28      0.15      1963
           6       0.04      0.08      0.05      1269
           7       0.06      0.34      0.10      1268
           8       0.04      0.09      0.06      1198
           9       0.05      0.12      0.07      1016
          10       0.58      0.03      0.05     19029

    accuracy                           0.13     41905
   macro avg       0.16      0.19      0.13     41905
weighted avg       0.35      0.13      0.12     41905

--------------------------------------------------
Results for MultinomialNB_BoW Training: accuracy=0.4088989117983963
Result

d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [41]:
results_POS_original

{'SVC_BoW StopWords removed': 0.13430378236487292,
 'MultinomialNB_BoW StopWords removed': 0.40746927574275144,
 'MLP StopWords removed': 0.4539076482519986}

#### HMM 

In [33]:
svd_POS = TruncatedSVD(n_components=20, random_state=42)
x_train_svd_POS = svd_POS.fit_transform(x_train_POS)
x_test_svd_POS = svd_POS.transform(x_test_POS)

scaler = StandardScaler()
x_train_svd_POS = scaler.fit_transform(x_train_svd_POS)
x_test_svd_POS = scaler.transform(x_test_svd_POS)

In [29]:
hmm_POS=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

In [35]:
lengths_train = [1] * x_train_svd_POS.shape[0]
lengths_test = [1] * x_test_svd_POS.shape[0]

In [34]:
hmm_POS.fit(x_train_svd_POS)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [39]:
y_pred_POS_original=hmm_POS.predict(x_train_svd_POS,)
accuracy_hmm_POS_original = accuracy_score(y_train, y_pred_POS_original)   
print(f"training HMM POS Accuracy : {accuracy_hmm_POS_original}")

training HMM POS Accuracy : 0.0362077605956472


In [37]:
y_pred_POS_original=hmm_POS.predict(x_test_svd_POS,lengths=lengths_test)
accuracy_hmm_POS_original = accuracy_score(y_test, y_pred_POS_original)   
print(f"testing HMM POS Accuracy with length : {accuracy_hmm_POS_original}")

testing HMM POS Accuracy with length : 0.022551008232907767


In [38]:
y_pred_POS_original=hmm_POS.predict(x_test_svd_POS,)
accuracy_hmm_POS_original = accuracy_score(y_test, y_pred_POS_original)   
print(f"testing HMM POS Accuracy : {accuracy_hmm_POS_original}")

testing HMM POS Accuracy : 0.03667820069204152


In [80]:
results_POS_original['HMM StopWords removed']=0.03667820069204152

### stopwords kept

In [42]:
for model_name, model in models_POS.items():
    model.fit(x_train_pos_stopkept_scaled, y_train)

    y_pred_train = model.predict(x_train_pos_stopkept_scaled)
    accuracy_train=accuracy_score(y_train, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_pos_stopkept_scaled)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)
    results_POS_original[model_name+" StopWords kept"] = accuracy

Results for SVC_POS Training: accuracy=0.1991635643375334
Results for SVC_POS Testing: accuracy=0.16735473093902875
              precision    recall  f1-score   support

           1       0.32      0.21      0.25      7120
           2       0.22      0.37      0.28      3589
           3       0.18      0.29      0.22      3473
           4       0.13      0.34      0.19      1980
           5       0.17      0.30      0.22      1963
           6       0.06      0.14      0.09      1269
           7       0.10      0.35      0.16      1268
           8       0.06      0.12      0.08      1198
           9       0.04      0.24      0.07      1016
          10       0.60      0.05      0.09     19029

    accuracy                           0.17     41905
   macro avg       0.19      0.24      0.16     41905
weighted avg       0.38      0.17      0.16     41905

--------------------------------------------------
Results for MultinomialNB_POS Training: accuracy=0.37417072355861014
Resul

d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#### HMM

In [ ]:
hmm_POS_stopkept=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

In [55]:
svd_POS_stopkept = TruncatedSVD(n_components=35, random_state=42)
x_train_svd_POS_stopkept = svd_POS_stopkept.fit_transform(x_train_POS_stopkept)
x_test_svd_POS_stopkept = svd_POS_stopkept.transform(x_test_POS_stopkept)

scaler_stopkept = StandardScaler()
x_train_svd_POS_stopkept_scaled = scaler_stopkept.fit_transform(x_train_svd_POS_stopkept)
x_test_svd_POS_stopkept_scaled = scaler_stopkept.transform(x_test_svd_POS_stopkept)

In [56]:
lengths_train = [1] * x_train_svd_POS_stopkept_scaled.shape[0]
lengths_test = [1] * x_test_svd_POS_stopkept_scaled.shape[0]

In [57]:
hmm_POS_stopkept.fit(x_train_svd_POS_stopkept_scaled)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [59]:
y_pred_POS_stopkept_original=hmm_POS_stopkept.predict(x_train_svd_POS_stopkept_scaled,)
accuracy_hmm_POS_stopkept_original = accuracy_score(y_train, y_pred_POS_stopkept_original)   
print(f"training HMM POS Accuracy : {accuracy_hmm_POS_stopkept_original}")

training HMM POS Accuracy : 0.05114666857579229


In [60]:
y_pred_POS_stopkept_original=hmm_POS_stopkept.predict(x_test_svd_POS_stopkept_scaled,lengths=lengths_test)
accuracy_hmm_POS_stopkept_original = accuracy_score(y_test, y_pred_POS_stopkept_original)   
print(f"testing HMM POS Accuracy : {accuracy_hmm_POS_stopkept_original}")

testing HMM POS Accuracy : 0.0


In [61]:
y_pred_POS_stopkept_original=hmm_POS_stopkept.predict(x_test_svd_POS_stopkept_scaled)
accuracy_hmm_POS_stopkept_original = accuracy_score(y_test, y_pred_POS_stopkept_original)   
print(f"testing HMM POS Accuracy : {accuracy_hmm_POS_stopkept_original}")

testing HMM POS Accuracy : 0.0530962892256294


In [81]:
results_POS_original['HMM StopWords kept']=0.0530962892256294

## With tf-idf

### stopwords removed

In [43]:
for model_name, model in models_POS.items():
    model.fit(x_train_pos_tfidf_scaled, y_train)

    y_pred_train = model.predict(x_train_pos_tfidf_scaled)
    accuracy_train=accuracy_score(y_train, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_pos_tfidf_scaled)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)
    results_POS_original["(tf-idf) "+model_name+" StopWords removed"] = accuracy

Results for SVC_POS Training: accuracy=0.1330541237113402
Results for SVC_POS Testing: accuracy=0.12208566996778428
              precision    recall  f1-score   support

           1       0.25      0.11      0.15      7120
           2       0.17      0.35      0.23      3589
           3       0.14      0.24      0.17      3473
           4       0.10      0.29      0.14      1980
           5       0.09      0.27      0.14      1963
           6       0.04      0.05      0.04      1269
           7       0.06      0.18      0.09      1268
           8       0.04      0.10      0.06      1198
           9       0.03      0.12      0.05      1016
          10       0.56      0.03      0.06     19029

    accuracy                           0.12     41905
   macro avg       0.15      0.18      0.11     41905
weighted avg       0.34      0.12      0.11     41905

--------------------------------------------------
Results for MultinomialNB_POS Training: accuracy=0.39814218213058417
Resul

d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#### HMM

In [74]:
svd_POS_tfidf = TruncatedSVD(n_components=20, random_state=42)
x_train_svd_POS_tfidf = svd_POS_tfidf.fit_transform(x_train_POS_tfidf)
x_test_svd_POS_tfidf = svd_POS_tfidf.transform(x_test_POS_tfidf)

scaler = StandardScaler()
x_train_svd_POS_tfidf_scaled = scaler.fit_transform(x_train_svd_POS_tfidf)
x_test_svd_POS_tfidf_scaled = scaler.transform(x_test_svd_POS_tfidf)

In [72]:
hmm_POS_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

In [75]:
lengths_train = [1] * x_train_svd_POS_tfidf_scaled.shape[0]
lengths_test = [1] * x_test_svd_POS_tfidf_scaled.shape[0]

In [76]:
hmm_POS_tfidf.fit(x_train_svd_POS_tfidf_scaled)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [77]:
y_pred_POS_original_tfidf=hmm_POS_tfidf.predict(x_train_svd_POS_tfidf_scaled,lengths=lengths_train)
accuracy_hmm_POS_original = accuracy_score(y_train, y_pred_POS_original_tfidf)   
print(f"training HMM POS Accuracy : {accuracy_hmm_POS_original}")

training HMM POS Accuracy : 0.04725085910652921


In [78]:
y_pred_POS_original_tfidf=hmm_POS_tfidf.predict(x_test_svd_POS_tfidf_scaled,lengths=lengths_test)
accuracy_hmm_POS_original = accuracy_score(y_test, y_pred_POS_original_tfidf)   
print(f"testing HMM POS Accuracy : {accuracy_hmm_POS_original}")

testing HMM POS Accuracy : 0.04724973153561628


In [79]:
y_pred_POS_original_tfidf=hmm_POS_tfidf.predict(x_test_svd_POS_tfidf_scaled,)
accuracy_hmm_POS_original = accuracy_score(y_test, y_pred_POS_original_tfidf)   
print(f"testing HMM POS Accuracy : {accuracy_hmm_POS_original}")

testing HMM POS Accuracy : 0.039995227299844886


In [82]:
results_POS_original['(tf-idf) HMM StopWords removed']=0.04724973153561628

### stopwords kept

In [14]:
for model_name, model in models_POS.items():
    model.fit(x_train_pos_stopkept_tfidf_scaled, y_train)

    y_pred_train = model.predict(x_train_pos_stopkept_tfidf_scaled)
    accuracy_train=accuracy_score(y_train, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_pos_stopkept_tfidf_scaled)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)
    results_POS_original['(tf-idf) '+model_name+" StopWords kept"] = accuracy

Results for SVC_POS Training: accuracy=0.18079419625811377
Results for SVC_POS Testing: accuracy=0.16050590621644195
              precision    recall  f1-score   support

           1       0.31      0.20      0.25      7120
           2       0.22      0.35      0.27      3589
           3       0.18      0.28      0.22      3473
           4       0.11      0.34      0.17      1980
           5       0.16      0.30      0.21      1963
           6       0.06      0.14      0.08      1269
           7       0.10      0.34      0.16      1268
           8       0.05      0.12      0.07      1198
           9       0.04      0.20      0.07      1016
          10       0.63      0.04      0.08     19029

    accuracy                           0.16     41905
   macro avg       0.19      0.23      0.16     41905
weighted avg       0.39      0.16      0.15     41905

--------------------------------------------------
Results for MultinomialNB_POS Training: accuracy=0.3668742840778923
Resul

d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [15]:
results_POS_original

{'SVC_BoW StopWords removed': 0.13430378236487292,
 'MultinomialNB_BoW StopWords removed': 0.40746927574275144,
 'MLP StopWords removed': 0.4539076482519986,
 'SVC_POS StopWords kept': 0.16735473093902875,
 'MultinomialNB_POS StopWords kept': 0.37441832716859563,
 'MLP StopWords kept': 0.45825080539315116,
 '(tf-idf) SVC_POS StopWords removed': 0.12208566996778428,
 '(tf-idf) MultinomialNB_POS StopWords removed': 0.39661138288986997,
 '(tf-idf) MLP StopWords removed': 0.45426560076363204,
 '(tf-idf) SVC_POS StopWords kept': 0.16050590621644195,
 '(tf-idf) MultinomialNB_POS StopWords kept': 0.36780813745376445,
 '(tf-idf) MLP StopWords kept': 0.4558644553155948,
 'HMM StopWords removed': 0.03667820069204152,
 'HMM StopWords kept': 0.0530962892256294,
 '(tf-idf) HMM StopWords removed': 0.04724973153561628,
 '(tf-idf) HMM StopWords kept': 0.08564610428349839}

#### HMM

In [16]:
hmm_POS_stopkept_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

In [17]:
svd_POS_stopkept_tfidf = TruncatedSVD(n_components=35, random_state=42)
x_train_svd_POS_stopkept_tfidf = svd_POS_stopkept_tfidf.fit_transform(x_train_POS_stopkept_tfidf)
x_test_svd_POS_stopkept_tfidf = svd_POS_stopkept_tfidf.transform(x_test_POS_stopkept_tfidf)

scaler_stopkept_tfidf = StandardScaler()
x_train_svd_POS_stopkept_tfidf_scaled = scaler_stopkept_tfidf.fit_transform(x_train_svd_POS_stopkept_tfidf)
x_test_svd_POS_stopkept_tfidf_scaled = scaler_stopkept_tfidf.transform(x_test_svd_POS_stopkept_tfidf)

In [18]:
lengths_train = [1] * x_train_svd_POS_stopkept_tfidf_scaled.shape[0]
lengths_test = [1] * x_test_svd_POS_stopkept_tfidf_scaled.shape[0]

In [19]:
hmm_POS_stopkept_tfidf.fit(x_train_svd_POS_stopkept_tfidf_scaled)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [20]:
y_pred_POS_stopkept_tfidf_original=hmm_POS_stopkept_tfidf.predict(x_train_svd_POS_stopkept_tfidf_scaled,lengths=lengths_train)
accuracy_hmm_POS_stopkept_tfidf_original = accuracy_score(y_train, y_pred_POS_stopkept_tfidf_original)   
print(f"training HMM POS Accuracy : {accuracy_hmm_POS_stopkept_tfidf_original}")

training HMM POS Accuracy : 0.08187762504772814


In [21]:
y_pred_POS_stopkept_tfidf_original=hmm_POS_stopkept_tfidf.predict(x_test_svd_POS_stopkept_tfidf_scaled,)
accuracy_hmm_POS_stopkept_tfidf_original = accuracy_score(y_test, y_pred_POS_stopkept_tfidf_original)   
print(f"testing HMM POS Accuracy : {accuracy_hmm_POS_stopkept_tfidf_original}")

testing HMM POS Accuracy : 0.06925187925068607


In [22]:
y_pred_POS_stopkept_tfidf_original=hmm_POS_stopkept_tfidf.predict(x_test_svd_POS_stopkept_tfidf_scaled,lengths=lengths_test)
accuracy_hmm_POS_stopkept_tfidf_original = accuracy_score(y_test, y_pred_POS_stopkept_tfidf_original)   
print(f"testing HMM POS Accuracy : {accuracy_hmm_POS_stopkept_tfidf_original}")

testing HMM POS Accuracy : 0.08111203913614128


In [24]:
results_POS_original['(tf-idf) HMM StopWords kept']=0.08111203913614128

## Save results

In [25]:
results_POS_original

{'SVC_BoW StopWords removed': 0.13430378236487292,
 'MultinomialNB_BoW StopWords removed': 0.40746927574275144,
 'MLP StopWords removed': 0.4539076482519986,
 'SVC_POS StopWords kept': 0.16735473093902875,
 'MultinomialNB_POS StopWords kept': 0.37441832716859563,
 'MLP StopWords kept': 0.45825080539315116,
 '(tf-idf) SVC_POS StopWords removed': 0.12208566996778428,
 '(tf-idf) MultinomialNB_POS StopWords removed': 0.39661138288986997,
 '(tf-idf) MLP StopWords removed': 0.45426560076363204,
 '(tf-idf) SVC_POS StopWords kept': 0.16050590621644195,
 '(tf-idf) MultinomialNB_POS StopWords kept': 0.36780813745376445,
 '(tf-idf) MLP StopWords kept': 0.4558644553155948,
 'HMM StopWords removed': 0.03667820069204152,
 'HMM StopWords kept': 0.0530962892256294,
 '(tf-idf) HMM StopWords removed': 0.04724973153561628,
 '(tf-idf) HMM StopWords kept': 0.08111203913614128}

In [26]:
joblib.dump(results_POS_original,'results_POS_original')

['results_POS_original']

In [86]:
results_POS_original=joblib.load('results_POS_original')
results_POS_original

{'SVC_BoW StopWords removed': 0.13430378236487292,
 'MultinomialNB_BoW StopWords removed': 0.40746927574275144,
 'MLP StopWords removed': 0.4539076482519986,
 'SVC_POS StopWords kept': 0.16735473093902875,
 'MultinomialNB_POS StopWords kept': 0.37441832716859563,
 'MLP StopWords kept': 0.45825080539315116,
 '(tf-idf) SVC_POS StopWords removed': 0.12208566996778428,
 '(tf-idf) MultinomialNB_POS StopWords removed': 0.39661138288986997,
 '(tf-idf) MLP StopWords removed': 0.45426560076363204,
 '(tf-idf) SVC_POS StopWords kept': 0.12208566996778428,
 '(tf-idf) MultinomialNB_POS StopWords kept': 0.39661138288986997,
 '(tf-idf) MLP StopWords kept': 0.45426560076363204,
 'HMM StopWords removed': 0.03667820069204152,
 'HMM StopWords kept': 0.0530962892256294,
 '(tf-idf) HMM StopWords removed': 0.04724973153561628,
 '(tf-idf) HMM StopWords kept': 0.08564610428349839}

In [27]:
with open("results_POS_original.txt", "w", encoding="utf-8") as f:
    for key, value in results_POS_original.items():
        f.write(f"{key}: {value}\n")

# ______________________________________________________________________________________

# resampled Data (undersample)

In [ ]:
# results_POS_undersample={}
results_POS_undersample=joblib.load('results_POS_undersample')
results_POS_undersample

{}

## BoW feature Extraction + Scaling

### with countvectorizer

In [24]:
vectorizer_POS_undersample = CountVectorizer(tokenizer=lambda x:pos_features(x, remove_stopwords=True))
x_train_POS_undersample = vectorizer_POS_undersample.fit_transform(x_train_undersampled)
x_test_POS_undersample= vectorizer_POS_undersample.transform(x_test_undersampled)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [25]:
scaler = StandardScaler(with_mean=False)
x_train_pos_undersample_scaled = scaler.fit_transform(x_train_POS_undersample)
x_test_pos_undersample_scaled  = scaler.transform(x_test_POS_undersample)

In [31]:
x_train_pos_undersample_scaled.shape,x_test_pos_undersample_scaled.shape

((66774, 35), (16694, 35))

In [26]:
vectorizer_POS_stopKept_undersample = CountVectorizer(tokenizer=lambda x:pos_features(x, remove_stopwords=False))
x_train_POS_stopKept_undersample = vectorizer_POS_stopKept_undersample.fit_transform(x_train_undersampled)
x_test_POS_stopKept_undersample= vectorizer_POS_stopKept_undersample.transform(x_test_undersampled)

In [27]:
scaler_stopKept = StandardScaler(with_mean=False)
x_train_pos_stopKept_undersample_scaled = scaler_stopKept.fit_transform(x_train_POS_stopKept_undersample)
x_test_pos_stopKept_undersample_scaled  = scaler_stopKept.transform(x_test_POS_stopKept_undersample)

In [30]:
x_train_pos_stopKept_undersample_scaled.shape,x_test_pos_stopKept_undersample_scaled.shape

((66774, 37), (16694, 37))

### with tf-idf vectorizer

In [13]:
vectorizer_POS_undersample_tfidf = TfidfVectorizer(tokenizer=lambda x:pos_features(x, remove_stopwords=True))
x_train_POS_undersample_tfidf = vectorizer_POS_undersample_tfidf.fit_transform(x_train_undersampled)
x_test_POS_undersample_tfidf= vectorizer_POS_undersample_tfidf.transform(x_test_undersampled)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [14]:
scaler_tfidf = StandardScaler(with_mean=False)
x_train_pos_undersample_scaled_tfidf = scaler_tfidf.fit_transform(x_train_POS_undersample_tfidf)
x_test_pos_undersample_scaled_tfidf  = scaler_tfidf.transform(x_test_POS_undersample_tfidf)

In [32]:
x_train_pos_undersample_scaled_tfidf.shape,x_test_pos_undersample_scaled_tfidf.shape

((66774, 35), (16694, 35))

In [15]:
vectorizer_POS_stopKept_undersample_tfidf = TfidfVectorizer(tokenizer=lambda x:pos_features(x, remove_stopwords=False))
x_train_POS_stopKept_undersample_tfidf = vectorizer_POS_stopKept_undersample_tfidf.fit_transform(x_train_undersampled)
x_test_POS_stopKept_undersample_tfidf= vectorizer_POS_stopKept_undersample_tfidf.transform(x_test_undersampled)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [16]:
scaler_stopKept_tfidf = StandardScaler(with_mean=False)
x_train_pos_stopKept_undersample_scaled_tfidf = scaler_stopKept_tfidf.fit_transform(x_train_POS_stopKept_undersample_tfidf)
x_test_pos_stopKept_undersample_scaled_tfidf  = scaler_stopKept_tfidf.transform(x_test_POS_stopKept_undersample_tfidf)

In [33]:
x_train_pos_stopKept_undersample_scaled_tfidf.shape,x_test_pos_stopKept_undersample_scaled_tfidf.shape

((66774, 37), (16694, 37))

## Models

In [34]:
models_POS_undersample={
    'SVM': SVC(),
    'MultinomialNB': MultinomialNB(),
    'MLP': MLPClassifier(hidden_layer_sizes=(32,16),activation='relu',solver='adam',max_iter=100,alpha=0.001,early_stopping=True,random_state=42),
}

## with countvectorizer

### stop words removed

In [40]:
for model_name, model in models_POS_undersample.items():
    model.fit(x_train_pos_undersample_scaled, y_train_undersampled)

    y_pred_train = model.predict(x_train_pos_undersample_scaled)
    accuracy_train=accuracy_score(y_train_undersampled, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_pos_undersample_scaled)
    accuracy=accuracy_score(y_test_undersampled, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test_undersampled, y_pred))
    print('-'*50)
    results_POS_undersample['undersample '+model_name+" StopWords removed"] = accuracy

Results for SVM Training: accuracy=0.23753257255818133
Results for SVM Testing: accuracy=0.22157661435246195
              precision    recall  f1-score   support

           1       0.18      0.26      0.21      2000
           2       0.27      0.41      0.32      2000
           3       0.21      0.33      0.26      2000
           4       0.23      0.43      0.30      1980
           5       0.26      0.32      0.28      1963
           6       0.15      0.00      0.00      1269
           7       0.31      0.01      0.02      1268
           8       0.12      0.00      0.00      1198
           9       0.44      0.00      0.01      1016
          10       0.16      0.10      0.12      2000

    accuracy                           0.22     16694
   macro avg       0.23      0.19      0.15     16694
weighted avg       0.23      0.22      0.18     16694

--------------------------------------------------
Results for MultinomialNB Training: accuracy=0.16749034055171175
Results for Mult

### stop words kept

In [38]:
for model_name, model in models_POS_undersample.items():
    model.fit(x_train_pos_stopKept_undersample_scaled, y_train_undersampled)

    y_pred_train = model.predict(x_train_pos_stopKept_undersample_scaled)
    accuracy_train=accuracy_score(y_train_undersampled, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_pos_stopKept_undersample_scaled)
    accuracy=accuracy_score(y_test_undersampled, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test_undersampled, y_pred))
    print('-'*50)
    results_POS_undersample['undersample '+model_name+" StopWords kept"] = accuracy

Results for SVM Training: accuracy=0.29839458471860303
Results for SVM Testing: accuracy=0.2605127590751168
              precision    recall  f1-score   support

           1       0.22      0.37      0.28      2000
           2       0.29      0.46      0.36      2000
           3       0.25      0.38      0.30      2000
           4       0.26      0.44      0.33      1980
           5       0.32      0.33      0.32      1963
           6       0.29      0.04      0.07      1269
           7       0.31      0.13      0.19      1268
           8       0.28      0.02      0.03      1198
           9       0.00      0.00      0.00      1016
          10       0.17      0.09      0.12      2000

    accuracy                           0.26     16694
   macro avg       0.24      0.23      0.20     16694
weighted avg       0.25      0.26      0.23     16694

--------------------------------------------------
Results for MultinomialNB Training: accuracy=0.1956899391978914
Results for Multin

### HMM

In [ ]:
hmm_POS_undersample=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_POS_stopkept_undersample=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

#### dimension reduction (66774,35)

In [47]:
svd_POS = TruncatedSVD(n_components=35, random_state=42)
x_train_svd_POS_undersample = svd_POS.fit_transform(x_train_pos_undersample_scaled)
x_test_svd_POS_undersample = svd_POS.transform(x_test_pos_undersample_scaled)

scaler = StandardScaler()
x_train_svd_POS_undersample = scaler.fit_transform(x_train_svd_POS_undersample)
x_test_svd_POS_undersample = scaler.transform(x_test_svd_POS_undersample)

In [48]:
svd_POS = TruncatedSVD(n_components=35, random_state=42)
x_train_svd_POS_Stopkept_undersample = svd_POS.fit_transform(x_train_pos_stopKept_undersample_scaled)
x_test_svd_POS_Stopkept_undersample = svd_POS.transform(x_test_pos_stopKept_undersample_scaled)

scaler = StandardScaler()
x_train_svd_POS_Stopkept_undersample = scaler.fit_transform(x_train_svd_POS_Stopkept_undersample)
x_test_svd_POS_Stopkept_undersample = scaler.transform(x_test_svd_POS_Stopkept_undersample)

#### apply HMM

##### stopwords removed

In [50]:
lengths_train = [1] * x_train_svd_POS_undersample.shape[0]
lengths_test = [1] * x_test_svd_POS_undersample.shape[0]

In [51]:
hmm_POS_undersample.fit(x_train_svd_POS_undersample)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [52]:
y_pred_stem_undersample=hmm_POS_undersample.predict(x_train_svd_POS_undersample,lengths=lengths_train)
accuracy_hmm_stem_undersample = accuracy_score(y_train_undersampled, y_pred_stem_undersample)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_stem_undersample}")

training HMM BoW Accuracy with lengths: 0.11980711055201126


In [53]:
y_pred_stem_undersample=hmm_POS_undersample.predict(x_test_svd_POS_undersample,lengths=lengths_test)
accuracy_hmm_stem_undersample = accuracy_score(y_test_undersampled, y_pred_stem_undersample)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_stem_undersample}")

testing HMM BoW Accuracy with lengths: 0.11980352222355337


In [54]:
y_pred_stem_undersample=hmm_POS_undersample.predict(x_test_svd_POS_undersample)
accuracy_hmm_stem_undersample = accuracy_score(y_test_undersampled, y_pred_stem_undersample)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_stem_undersample}")

testing HMM BoW Accuracy: 0.06870731999520786


In [55]:
results_POS_undersample['undersample HMM StopWords removed']=0.11980352222355337

##### stopwords kept

In [64]:
hmm_POS_stopkept_undersample.fit(x_train_svd_POS_Stopkept_undersample)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [65]:
y_pred_POS_stopkept_undersample=hmm_POS_stopkept_undersample.predict(x_train_svd_POS_Stopkept_undersample,lengths=lengths_train)
accuracy_hmm_POS_stopkept_undersample = accuracy_score(y_train_undersampled, y_pred_POS_stopkept_undersample)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_POS_stopkept_undersample}")

training HMM BoW Accuracy with lengths: 0.11890855722287118


In [66]:
y_pred_POS_stopkept_undersample=hmm_POS_stopkept_undersample.predict(x_test_svd_POS_Stopkept_undersample,lengths=lengths_test)
accuracy_hmm_POS_stopkept_undersample = accuracy_score(y_test_undersampled, y_pred_POS_stopkept_undersample)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_POS_stopkept_undersample}")

testing HMM BoW Accuracy with lengths: 0.1189648975679885


In [67]:
y_pred_POS_stopkept_undersample=hmm_POS_stopkept_undersample.predict(x_test_svd_POS_Stopkept_undersample,)
accuracy_hmm_POS_stopkept_undersample = accuracy_score(y_test_undersampled, y_pred_POS_stopkept_undersample)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_POS_stopkept_undersample}")

testing HMM BoW Accuracy: 0.08020845812866899


In [68]:
results_POS_undersample['undersample HMM StopWords kept']=0.1189648975679885

#### save the results of HMM

In [69]:
results_POS_undersample

{'undersample SVM StopWords removed': 0.22157661435246195,
 'undersample MultinomialNB StopWords removed': 0.16874326105187493,
 'undersample MLP StopWords removed': 0.2240924883191566,
 'undersample SVM StopWords kept': 0.2605127590751168,
 'undersample MultinomialNB StopWords kept': 0.19803522223553371,
 'undersample MLP StopWords kept': 0.25410327063615673,
 '(tf-idf) undersample SVM StopWords removed': 0.20672097759674135,
 '(tf-idf) undersample MultinomialNB StopWords removed': 0.1759913741463999,
 '(tf-idf) undersample MLP StopWords removed': 0.20779920929675333,
 '(tf-idf) undersample SVM StopWords kept': 0.2509883790583443,
 '(tf-idf) undersample MultinomialNB StopWords kept': 0.2001916856355577,
 '(tf-idf) undersample MLP StopWords kept': 0.24547741703606085,
 'undersample HMM StopWords removed': 0.11980352222355337,
 'undersample HMM StopWords kept': 0.1189648975679885}

In [70]:
joblib.dump(results_POS_undersample,'results_POS_undersample')

['results_POS_undersample']

## with tf-idf

### stop words removed

In [41]:
for model_name, model in models_POS_undersample.items():
    model.fit(x_train_pos_undersample_scaled_tfidf, y_train_undersampled)

    y_pred_train = model.predict(x_train_pos_undersample_scaled_tfidf)
    accuracy_train=accuracy_score(y_train_undersampled, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_pos_undersample_scaled_tfidf)
    accuracy=accuracy_score(y_test_undersampled, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test_undersampled, y_pred))
    print('-'*50)
    results_POS_undersample['(tf-idf) undersample '+model_name+" StopWords removed"] = accuracy

Results for SVM Training: accuracy=0.21854314553568754
Results for SVM Testing: accuracy=0.20672097759674135
              precision    recall  f1-score   support

           1       0.18      0.23      0.20      2000
           2       0.23      0.43      0.30      2000
           3       0.19      0.27      0.23      2000
           4       0.21      0.42      0.29      1980
           5       0.22      0.31      0.26      1963
           6       0.11      0.00      0.00      1269
           7       0.28      0.01      0.02      1268
           8       0.11      0.00      0.00      1198
           9       0.50      0.00      0.01      1016
          10       0.15      0.07      0.10      2000

    accuracy                           0.21     16694
   macro avg       0.22      0.17      0.14     16694
weighted avg       0.21      0.21      0.16     16694

--------------------------------------------------
Results for MultinomialNB Training: accuracy=0.1767154880642166
Results for Multi

d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### stop words kept

In [43]:
for model_name, model in models_POS_undersample.items():
    model.fit(x_train_pos_stopKept_undersample_scaled_tfidf, y_train_undersampled)

    y_pred_train = model.predict(x_train_pos_stopKept_undersample_scaled_tfidf)
    accuracy_train=accuracy_score(y_train_undersampled, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_pos_stopKept_undersample_scaled_tfidf)
    accuracy=accuracy_score(y_test_undersampled, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test_undersampled, y_pred))
    print('-'*50)
    results_POS_undersample['(tf-idf) undersample '+model_name+" StopWords kept"] = accuracy

Results for SVM Training: accuracy=0.27611046215592894
Results for SVM Testing: accuracy=0.2509883790583443
              precision    recall  f1-score   support

           1       0.22      0.36      0.28      2000
           2       0.28      0.42      0.34      2000
           3       0.25      0.35      0.29      2000
           4       0.24      0.42      0.31      1980
           5       0.30      0.34      0.32      1963
           6       0.30      0.03      0.06      1269
           7       0.31      0.14      0.20      1268
           8       0.22      0.01      0.03      1198
           9       0.00      0.00      0.00      1016
          10       0.16      0.09      0.12      2000

    accuracy                           0.25     16694
   macro avg       0.23      0.22      0.19     16694
weighted avg       0.24      0.25      0.22     16694

--------------------------------------------------
Results for MultinomialNB Training: accuracy=0.1969628897475065
Results for Multin

d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Results for MLP Training: accuracy=0.25216401593434573
Results for MLP Testing: accuracy=0.24547741703606085
              precision    recall  f1-score   support

           1       0.21      0.33      0.26      2000
           2       0.27      0.44      0.34      2000
           3       0.23      0.36      0.28      2000
           4       0.26      0.35      0.30      1980
           5       0.28      0.38      0.32      1963
           6       0.24      0.05      0.08      1269
           7       0.29      0.16      0.20      1268
           8       0.23      0.01      0.01      1198
           9       0.21      0.00      0.01      1016
          10       0.17      0.06      0.09      2000

    accuracy                           0.25     16694
   macro avg       0.24      0.21      0.19     16694
weighted avg       0.24      0.25      0.21     16694

--------------------------------------------------


In [44]:
results_POS_undersample

{'undersample SVM StopWords removed': 0.22157661435246195,
 'undersample MultinomialNB StopWords removed': 0.16874326105187493,
 'undersample MLP StopWords removed': 0.2240924883191566,
 'undersample SVM StopWords kept': 0.2605127590751168,
 'undersample MultinomialNB StopWords kept': 0.19803522223553371,
 'undersample MLP StopWords kept': 0.25410327063615673,
 '(tf-idf) undersample SVM StopWords removed': 0.20672097759674135,
 '(tf-idf) undersample MultinomialNB StopWords removed': 0.1759913741463999,
 '(tf-idf) undersample MLP StopWords removed': 0.20779920929675333,
 '(tf-idf) undersample SVM StopWords kept': 0.2509883790583443,
 '(tf-idf) undersample MultinomialNB StopWords kept': 0.2001916856355577,
 '(tf-idf) undersample MLP StopWords kept': 0.24547741703606085}

### HMM

In [76]:
hmm_POS_undersample_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_POS_stopkept_undersample_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

#### dimension reduction (66774,50)

In [ ]:
svd_POS = TruncatedSVD(n_components=35, random_state=42)
x_train_svd_POS_undersample_tfidf = svd_POS.fit_transform(x_train_pos_undersample_scaled_tfidf)
x_test_svd_POS_undersample_tfidf = svd_POS.transform(x_test_pos_undersample_scaled_tfidf)

scaler = StandardScaler()
x_train_svd_POS_undersample_tfidf = scaler.fit_transform(x_train_svd_POS_undersample_tfidf)
x_test_svd_POS_undersample_tfidf = scaler.transform(x_test_svd_POS_undersample_tfidf)

In [72]:
svd_POS = TruncatedSVD(n_components=35, random_state=42)
x_train_svd_POS_Stopkept_undersample_tfidf = svd_POS.fit_transform(x_train_pos_stopKept_undersample_scaled_tfidf)
x_test_svd_POS_Stopkept_undersample_tfidf = svd_POS.transform(x_test_pos_stopKept_undersample_scaled_tfidf)

scaler = StandardScaler()
x_train_svd_POS_Stopkept_undersample_tfidf = scaler.fit_transform(x_train_svd_POS_Stopkept_undersample_tfidf)
x_test_svd_POS_Stopkept_undersample_tfidf = scaler.transform(x_test_svd_POS_Stopkept_undersample_tfidf)

#### apply HMM

##### stopwords removed

In [73]:
lengths_train = [1]*x_train_svd_POS_undersample_tfidf.shape[0]
lengths_test = [1]*x_test_svd_POS_undersample_tfidf.shape[0]

In [77]:
hmm_POS_undersample_tfidf.fit(x_train_svd_POS_undersample_tfidf)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [79]:
y_pred_stem_tfidf_undersample=hmm_POS_undersample_tfidf.predict(x_train_svd_POS_undersample_tfidf)
accuracy_hmm_stem_tfidf_undersample = accuracy_score(y_train_undersampled, y_pred_stem_tfidf_undersample)   
print(f"training HMM BoW Accuracy : {accuracy_hmm_stem_tfidf_undersample}")

training HMM BoW Accuracy : 0.11784526911672208


In [80]:
y_pred_stem_tfidf_undersample=hmm_POS_undersample_tfidf.predict(x_test_svd_POS_undersample_tfidf,lengths=lengths_test)
accuracy_hmm_stem_tfidf_undersample = accuracy_score(y_test_undersampled, y_pred_stem_tfidf_undersample)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_stem_tfidf_undersample}")

testing HMM BoW Accuracy with lengths: 0.11980352222355337


In [81]:
y_pred_stem_tfidf_undersample=hmm_POS_undersample_tfidf.predict(x_test_svd_POS_undersample_tfidf)
accuracy_hmm_stem_tfidf_undersample = accuracy_score(y_test_undersampled, y_pred_stem_tfidf_undersample)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_stem_tfidf_undersample}")

testing HMM BoW Accuracy: 0.11860548700131784


In [87]:
results_POS_undersample['(tf-idf)undersample HMM StopWords removed']=0.11980352222355337

##### stopwords kept

In [83]:
lengths_train = [1]*x_train_svd_POS_Stopkept_undersample_tfidf.shape[0]
lengths_test = [1]*x_test_svd_POS_Stopkept_undersample_tfidf.shape[0]

In [84]:
hmm_POS_stopkept_undersample_tfidf.fit(x_train_svd_POS_Stopkept_undersample_tfidf)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [91]:
y_pred_stem_stopkept_tfidf_undersample=hmm_POS_stopkept_undersample_tfidf.predict(x_train_svd_POS_Stopkept_undersample_tfidf,)
accuracy_hmm_stem_stopkept_tfidf_undersample = accuracy_score(y_train_undersampled, y_pred_stem_stopkept_tfidf_undersample)   
print(f"training HMM BoW Accuracy : {accuracy_hmm_stem_stopkept_tfidf_undersample}")

training HMM BoW Accuracy : 0.106373738281367


In [92]:
y_pred_stem_stopkept_tfidf_undersample=hmm_POS_stopkept_undersample_tfidf.predict(x_test_svd_POS_Stopkept_undersample_tfidf,lengths=lengths_test)
accuracy_hmm_stem_stopkept_tfidf_undersample = accuracy_score(y_test_undersampled, y_pred_stem_stopkept_tfidf_undersample)   
print(f"testing HMM BoW Accuracy : {accuracy_hmm_stem_stopkept_tfidf_undersample}")

testing HMM BoW Accuracy : 0.0034743021444830478


In [93]:
y_pred_stem_stopkept_tfidf_undersample=hmm_POS_stopkept_undersample_tfidf.predict(x_test_svd_POS_Stopkept_undersample_tfidf,)
accuracy_hmm_stem_stopkept_tfidf_undersample = accuracy_score(y_test_undersampled, y_pred_stem_stopkept_tfidf_undersample)   
print(f"testing HMM BoW Accuracy : {accuracy_hmm_stem_stopkept_tfidf_undersample}")

testing HMM BoW Accuracy : 0.10716425062896849


In [94]:
results_POS_undersample['(tf-idf) undersample HMM StopWords kept']=0.10716425062896849

### save the results

In [95]:
joblib.dump(results_POS_undersample,'results_POS_undersample')

['results_POS_undersample']

In [96]:
with open("results_POS_undersample.txt", "w", encoding="utf-8") as f:
    for key, value in results_POS_undersample.items():
        f.write(f"{key}: {value}\n")

In [97]:
results_POS_undersample

{'undersample SVM StopWords removed': 0.22157661435246195,
 'undersample MultinomialNB StopWords removed': 0.16874326105187493,
 'undersample MLP StopWords removed': 0.2240924883191566,
 'undersample SVM StopWords kept': 0.2605127590751168,
 'undersample MultinomialNB StopWords kept': 0.19803522223553371,
 'undersample MLP StopWords kept': 0.25410327063615673,
 '(tf-idf) undersample SVM StopWords removed': 0.20672097759674135,
 '(tf-idf) undersample MultinomialNB StopWords removed': 0.1759913741463999,
 '(tf-idf) undersample MLP StopWords removed': 0.20779920929675333,
 '(tf-idf) undersample SVM StopWords kept': 0.2509883790583443,
 '(tf-idf) undersample MultinomialNB StopWords kept': 0.2001916856355577,
 '(tf-idf) undersample MLP StopWords kept': 0.24547741703606085,
 'undersample HMM StopWords removed': 0.11980352222355337,
 'undersample HMM StopWords kept': 0.1189648975679885,
 '(tf-idf)undersample HMM StopWords removed': 0.11980352222355337,
 '(tf-idf) undersample HMM StopWords kep

# ______________________________________________________________________________________

# resampled Data (rosrus)

In [98]:
results_POS_rosrus={}

## BoW feature Extraction + scaling

### with countvictorizer

In [99]:
vectorizer_POS_rosrus = CountVectorizer(tokenizer=lambda x:pos_features(x, remove_stopwords=True))
x_train_POS_rosrus = vectorizer_POS_rosrus.fit_transform(x_train_bal)
x_test_POS_rosrus= vectorizer_POS_rosrus.transform(x_test_bal)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [100]:
scaler_rosrus = StandardScaler(with_mean=False)
x_train_pos_rosrus_scaled = scaler_rosrus.fit_transform(x_train_POS_rosrus)
x_test_pos_rosrus_scaled  = scaler_rosrus.transform(x_test_POS_rosrus)

In [101]:
x_train_pos_rosrus_scaled.shape,x_test_pos_rosrus_scaled.shape

((130000, 35), (41905, 35))

In [102]:
vectorizer_POS_stopkept_rosrus = CountVectorizer(tokenizer=lambda x:pos_features(x, remove_stopwords=False))
x_train_POS_stopkept_rosrus = vectorizer_POS_stopkept_rosrus.fit_transform(x_train_bal)
x_test_POS_stopkept_rosrus= vectorizer_POS_stopkept_rosrus.transform(x_test_bal)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [103]:
scaler_stopkept_rosrus = StandardScaler(with_mean=False)
x_train_pos_stopkept_rosrus_scaled = scaler_stopkept_rosrus.fit_transform(x_train_POS_stopkept_rosrus)
x_test_pos_stopkept_rosrus_scaled  = scaler_stopkept_rosrus.transform(x_test_POS_stopkept_rosrus)

In [104]:
x_train_pos_stopkept_rosrus_scaled.shape,x_test_pos_stopkept_rosrus_scaled.shape

((130000, 36), (41905, 36))

### with tf-idf

In [105]:
vectorizer_POS_rosrus_tfidf = TfidfVectorizer(tokenizer=lambda x:pos_features(x, remove_stopwords=True))
x_train_POS_rosrus_tfidf = vectorizer_POS_rosrus_tfidf.fit_transform(x_train_bal)
x_test_POS_rosrus_tfidf= vectorizer_POS_rosrus_tfidf.transform(x_test_bal)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [106]:
scaler_rosrus_tfidf = StandardScaler(with_mean=False)
x_train_pos_rosrus_tfidf_scaled = scaler_rosrus_tfidf.fit_transform(x_train_POS_rosrus_tfidf)
x_test_pos_rosrus_tfidf_scaled  = scaler_rosrus_tfidf.transform(x_test_POS_rosrus_tfidf)

In [107]:
x_train_pos_rosrus_tfidf_scaled.shape,x_test_pos_rosrus_tfidf_scaled.shape

((130000, 35), (41905, 35))

In [108]:
vectorizer_POS_stopkept_rosrus_tfidf = TfidfVectorizer(tokenizer=lambda x:pos_features(x, remove_stopwords=False))
x_train_POS_stopkept_rosrus_tfidf = vectorizer_POS_stopkept_rosrus_tfidf.fit_transform(x_train_bal)
x_test_POS_stopkept_rosrus_tfidf= vectorizer_POS_stopkept_rosrus_tfidf.transform(x_test_bal)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [109]:
scaler_stopkept_rosrus_tfidf = StandardScaler(with_mean=False)
x_train_pos_stopkept_rosrus_tfidf_scaled = scaler_stopkept_rosrus_tfidf.fit_transform(x_train_POS_stopkept_rosrus_tfidf)
x_test_pos_stopkept_rosrus_tfidf_scaled  = scaler_stopkept_rosrus_tfidf.transform(x_test_POS_stopkept_rosrus_tfidf)

In [110]:
x_train_pos_stopkept_rosrus_tfidf_scaled.shape,x_test_pos_stopkept_rosrus_tfidf_scaled.shape

((130000, 36), (41905, 36))

## Models

In [112]:
models_POS_rosrus={
    'SVM': SVC(),
    'MultinomialNB': MultinomialNB(),
    'MLP': MLPClassifier(hidden_layer_sizes=(32,16),activation='relu',solver='adam',max_iter=100,alpha=0.001,early_stopping=True,random_state=42),
}

### with countvectorizer

#### stopwords removed

In [ ]:
for model_name, model in models_POS_rosrus.items():
    model.fit(x_train_pos_rosrus_scaled, y_train_bal)

    y_pred_train = model.predict(x_train_pos_rosrus_scaled)
    accuracy_train=accuracy_score(y_train_bal, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_pos_rosrus_scaled)
    accuracy=accuracy_score(y_test_bal, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test_bal, y_pred))
    print('-'*50)
    results_POS_rosrus['rorus '+model_name+" StopWords removed"] = accuracy

Results for SVM Training: accuracy=0.22475384615384617
Results for SVM Testing: accuracy=0.1299128982221692
              precision    recall  f1-score   support

           1       0.24      0.14      0.17      7120
           2       0.20      0.33      0.25      3589
           3       0.15      0.28      0.19      3473
           4       0.11      0.29      0.16      1980
           5       0.11      0.27      0.15      1963
           6       0.04      0.08      0.05      1269
           7       0.06      0.30      0.10      1268
           8       0.04      0.08      0.05      1198
           9       0.04      0.12      0.06      1016
          10       0.56      0.03      0.05     19029

    accuracy                           0.13     41905
   macro avg       0.15      0.19      0.12     41905
weighted avg       0.34      0.13      0.11     41905

--------------------------------------------------
Results for MultinomialNB Training: accuracy=0.1461230769230769
Results for Multin

#### stopwords kept

In [ ]:
for model_name, model in models_POS_rosrus.items():
    model.fit(x_train_pos_stopkept_rosrus_scaled, y_train_bal)

    y_pred_train = model.predict(x_train_pos_stopkept_rosrus_scaled)
    accuracy_train=accuracy_score(y_train_bal, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_pos_stopkept_rosrus_scaled)
    accuracy=accuracy_score(y_test_bal, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test_bal, y_pred))
    print('-'*50)
    results_POS_rosrus['rorus '+model_name+" StopWords kept"] = accuracy

Results for SVM Training: accuracy=0.3147384615384615
Results for SVM Testing: accuracy=0.16618541940102613
              precision    recall  f1-score   support

           1       0.32      0.22      0.26      7120
           2       0.22      0.38      0.28      3589
           3       0.18      0.28      0.22      3473
           4       0.12      0.32      0.18      1980
           5       0.16      0.28      0.21      1963
           6       0.06      0.14      0.09      1269
           7       0.10      0.35      0.16      1268
           8       0.06      0.12      0.08      1198
           9       0.04      0.25      0.07      1016
          10       0.59      0.05      0.09     19029

    accuracy                           0.17     41905
   macro avg       0.19      0.24      0.16     41905
weighted avg       0.38      0.17      0.16     41905

--------------------------------------------------
Results for MultinomialNB Training: accuracy=0.17443076923076922
Results for Multi

#### HMM

In [162]:
hmm_POS_rosrus=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_POS_stopkept_rosrus=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

##### dimension reduction (,35)

In [163]:
svd_POS = TruncatedSVD(n_components=35, random_state=42)
x_train_svd_POS_rosrus = svd_POS.fit_transform(x_train_pos_rosrus_scaled)
x_test_svd_POS_rosrus = svd_POS.transform(x_test_pos_rosrus_scaled)

scaler = StandardScaler()
x_train_svd_POS_rosrus = scaler.fit_transform(x_train_svd_POS_rosrus)
x_test_svd_POS_rosrus = scaler.transform(x_test_svd_POS_rosrus)

In [164]:
svd_POS = TruncatedSVD(n_components=35, random_state=42)
x_train_svd_POS_Stopkept_rosrus = svd_POS.fit_transform(x_train_pos_stopkept_rosrus_scaled)
x_test_svd_POS_Stopkept_rosrus = svd_POS.transform(x_test_pos_stopkept_rosrus_scaled)

scaler = StandardScaler()
x_train_svd_POS_Stopkept_rosrus = scaler.fit_transform(x_train_svd_POS_Stopkept_rosrus)
x_test_svd_POS_Stopkept_rosrus = scaler.transform(x_test_svd_POS_Stopkept_rosrus)

##### stopwords removed

In [165]:
lengths_train = [1]*x_train_svd_POS_rosrus.shape[0]
lengths_test = [1]*x_test_svd_POS_rosrus.shape[0]

In [166]:
hmm_POS_rosrus.fit(x_train_svd_POS_rosrus)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [172]:
y_pred_POS_rosrus=hmm_POS_rosrus.predict(x_train_svd_POS_rosrus,lengths=lengths_train)
accuracy_hmm_POS_rosrus = accuracy_score(y_train_bal, y_pred_POS_rosrus)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_POS_rosrus}")

training HMM BoW Accuracy with lengths: 0.1


In [169]:
y_pred_POS_rosrus=hmm_POS_rosrus.predict(x_test_svd_POS_rosrus,lengths=lengths_test)
accuracy_hmm_POS_rosrus = accuracy_score(y_test_bal, y_pred_POS_rosrus)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_POS_rosrus}")

testing HMM BoW Accuracy with lengths: 0.08564610428349839


In [170]:
y_pred_POS_rosrus=hmm_POS_rosrus.predict(x_test_svd_POS_rosrus)
accuracy_hmm_POS_rosrus = accuracy_score(y_test_bal, y_pred_POS_rosrus)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_POS_rosrus}")

testing HMM BoW Accuracy: 0.09540627610070397


In [174]:
results_POS_rosrus['rosrus HMM StopWords removed']=0.09540627610070397

##### stopwords kept

In [175]:
lengths_train = [1]*x_train_svd_POS_Stopkept_rosrus.shape[0]
lengths_test = [1]*x_test_svd_POS_Stopkept_rosrus.shape[0]

In [176]:
hmm_POS_stopkept_rosrus.fit(x_train_svd_POS_Stopkept_rosrus)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [177]:
y_pred_POS_stopkept_rosrus=hmm_POS_stopkept_rosrus.predict(x_train_svd_POS_Stopkept_rosrus,lengths=lengths_train)
accuracy_hmm_POS_rosrus = accuracy_score(y_train_bal, y_pred_POS_stopkept_rosrus)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_POS_rosrus}")

training HMM BoW Accuracy with lengths: 0.10025384615384615


In [178]:
y_pred_POS_stopkept_rosrus=hmm_POS_stopkept_rosrus.predict(x_test_svd_POS_Stopkept_rosrus,lengths=lengths_test)
accuracy_hmm_POS_rosrus = accuracy_score(y_test_bal, y_pred_POS_stopkept_rosrus)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_POS_rosrus}")

testing HMM BoW Accuracy with lengths: 0.02586803484071113


In [179]:
y_pred_POS_stopkept_rosrus=hmm_POS_stopkept_rosrus.predict(x_test_svd_POS_Stopkept_rosrus,)
accuracy_hmm_POS_rosrus = accuracy_score(y_test_bal, y_pred_POS_stopkept_rosrus)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_POS_rosrus}")

testing HMM BoW Accuracy: 0.019997613649922443


In [180]:
results_POS_rosrus['rosrus HMM StopWords kept']=0.02586803484071113

### with tf-idf

#### stopwords removed

In [ ]:
for model_name, model in models_POS_rosrus.items():
    model.fit(x_train_pos_rosrus_tfidf_scaled, y_train_bal)

    y_pred_train = model.predict(x_train_pos_rosrus_tfidf_scaled)
    accuracy_train=accuracy_score(y_train_bal, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_pos_rosrus_tfidf_scaled)
    accuracy=accuracy_score(y_test_bal, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test_bal, y_pred))
    print('-'*50)
    results_POS_rosrus['(tf-idf) rorus '+model_name+" StopWords removed"] = accuracy

Results for SVM Training: accuracy=0.20159230769230768
Results for SVM Testing: accuracy=0.1185777353537764
              precision    recall  f1-score   support

           1       0.25      0.11      0.15      7120
           2       0.17      0.35      0.23      3589
           3       0.13      0.23      0.17      3473
           4       0.10      0.30      0.15      1980
           5       0.09      0.26      0.14      1963
           6       0.03      0.04      0.04      1269
           7       0.06      0.19      0.09      1268
           8       0.04      0.09      0.06      1198
           9       0.03      0.14      0.05      1016
          10       0.57      0.03      0.05     19029

    accuracy                           0.12     41905
   macro avg       0.15      0.17      0.11     41905
weighted avg       0.34      0.12      0.10     41905

--------------------------------------------------
Results for MultinomialNB Training: accuracy=0.14913846153846153
Results for Multi

#### stopwords kept

In [ ]:
for model_name, model in models_POS_rosrus.items():
    model.fit(x_train_pos_stopkept_rosrus_tfidf_scaled, y_train_bal)

    y_pred_train = model.predict(x_train_pos_stopkept_rosrus_tfidf_scaled)
    accuracy_train=accuracy_score(y_train_bal, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_pos_stopkept_rosrus_tfidf_scaled)
    accuracy=accuracy_score(y_test_bal, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test_bal, y_pred))
    print('-'*50)
    results_POS_rosrus['(tf-idf) rorus '+model_name+" StopWords kept"] = accuracy

Results for SVM Training: accuracy=0.28246923076923075
Results for SVM Testing: accuracy=0.16098317623195324
              precision    recall  f1-score   support

           1       0.32      0.22      0.26      7120
           2       0.21      0.37      0.27      3589
           3       0.18      0.27      0.21      3473
           4       0.12      0.33      0.17      1980
           5       0.15      0.28      0.20      1963
           6       0.06      0.14      0.08      1269
           7       0.10      0.33      0.16      1268
           8       0.06      0.12      0.08      1198
           9       0.04      0.21      0.06      1016
          10       0.59      0.04      0.08     19029

    accuracy                           0.16     41905
   macro avg       0.18      0.23      0.16     41905
weighted avg       0.37      0.16      0.15     41905

--------------------------------------------------
Results for MultinomialNB Training: accuracy=0.18008461538461537
Results for Mult

#### HMM

In [181]:
hmm_POS_rosrus_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_POS_stopkept_rosrus_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

##### dimension reduction (,35)

In [182]:
svd_POS_tfidf = TruncatedSVD(n_components=35, random_state=42)
x_train_svd_POS_tfidf_rosrus = svd_POS_tfidf.fit_transform(x_train_pos_rosrus_tfidf_scaled)
x_test_svd_POS_tfidf_rosrus = svd_POS_tfidf.transform(x_test_pos_rosrus_tfidf_scaled)

scaler = StandardScaler()
x_train_svd_POS_tfidf_rosrus = scaler.fit_transform(x_train_svd_POS_tfidf_rosrus)
x_test_svd_POS_tfidf_rosrus = scaler.transform(x_test_svd_POS_tfidf_rosrus)

In [183]:
svd_POS_tfidf = TruncatedSVD(n_components=35, random_state=42)
x_train_svd_POS_tfidf_Stopkept_rosrus = svd_POS_tfidf.fit_transform(x_train_pos_stopkept_rosrus_tfidf_scaled)
x_test_svd_POS_tfidf_Stopkept_rosrus = svd_POS_tfidf.transform(x_test_pos_stopkept_rosrus_tfidf_scaled)

scaler = StandardScaler()
x_train_svd_POS_tfidf_Stopkept_rosrus = scaler.fit_transform(x_train_svd_POS_tfidf_Stopkept_rosrus)
x_test_svd_POS_tfidf_Stopkept_rosrus = scaler.transform(x_test_svd_POS_tfidf_Stopkept_rosrus)

##### stopwords removed

In [184]:
lengths_train = [1]*x_train_svd_POS_tfidf_rosrus.shape[0]
lengths_test = [1]*x_test_svd_POS_tfidf_rosrus.shape[0]

In [185]:
hmm_POS_rosrus_tfidf.fit(x_train_svd_POS_tfidf_rosrus)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [186]:
y_pred_POS_rosrus_tfidf=hmm_POS_rosrus_tfidf.predict(x_train_svd_POS_tfidf_rosrus,lengths=lengths_train)
accuracy_hmm_POS_rosrus_tfidf = accuracy_score(y_train_bal, y_pred_POS_rosrus_tfidf)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_POS_rosrus_tfidf}")

training HMM BoW Accuracy with lengths: 0.0


In [187]:
y_pred_POS_rosrus_tfidf=hmm_POS_rosrus_tfidf.predict(x_test_svd_POS_tfidf_rosrus,lengths=lengths_test)
accuracy_hmm_POS_rosrus_tfidf = accuracy_score(y_test_bal, y_pred_POS_rosrus_tfidf)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_POS_rosrus_tfidf}")

testing HMM BoW Accuracy with lengths: 0.0


In [188]:
y_pred_POS_rosrus_tfidf=hmm_POS_rosrus_tfidf.predict(x_test_svd_POS_tfidf_rosrus,)
accuracy_hmm_POS_rosrus_tfidf = accuracy_score(y_test_bal, y_pred_POS_rosrus_tfidf)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_POS_rosrus_tfidf}")

testing HMM BoW Accuracy: 0.04827586206896552


In [189]:
results_POS_rosrus['(tf-idf) rosrus HMM StopWords removed']=0.04827586206896552

##### stopwords kept

In [192]:
lengths_train = [1]*x_train_svd_POS_tfidf_Stopkept_rosrus.shape[0]
lengths_test = [1]*x_test_svd_POS_tfidf_Stopkept_rosrus.shape[0]

In [193]:
hmm_POS_stopkept_rosrus_tfidf.fit(x_train_svd_POS_tfidf_Stopkept_rosrus)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [198]:
y_pred_POS_rosrus_tfidf=hmm_POS_stopkept_rosrus_tfidf.predict(x_train_svd_POS_tfidf_Stopkept_rosrus,lengths=lengths_train)
accuracy_hmm_POS_rosrus_tfidf = accuracy_score(y_train_bal, y_pred_POS_rosrus_tfidf)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_POS_rosrus_tfidf}")

training HMM BoW Accuracy with lengths: 0.1


In [195]:
y_pred_POS_rosrus_tfidf=hmm_POS_stopkept_rosrus_tfidf.predict(x_test_svd_POS_tfidf_Stopkept_rosrus,lengths=lengths_test)
accuracy_hmm_POS_rosrus_tfidf = accuracy_score(y_test_bal, y_pred_POS_rosrus_tfidf)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_POS_rosrus_tfidf}")

testing HMM BoW Accuracy with lengths: 0.030282782484190432


In [196]:
y_pred_POS_rosrus_tfidf=hmm_POS_stopkept_rosrus_tfidf.predict(x_test_svd_POS_tfidf_Stopkept_rosrus)
accuracy_hmm_POS_rosrus_tfidf = accuracy_score(y_test_bal, y_pred_POS_rosrus_tfidf)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_POS_rosrus_tfidf}")

testing HMM BoW Accuracy: 0.06199737501491469


In [199]:
results_POS_rosrus['(tf-idf) rosrus HMM StopWords kept']=0.06199737501491469

## Save results

In [200]:
joblib.dump(results_POS_rosrus,'results_POS_rosrus')

['results_POS_rosrus']

In [201]:
results_POS_rosrus

{'rorus SVM StopWords removed': 0.1299128982221692,
 'rorus MultinomialNB StopWords removed': 0.10485622240782723,
 'rorus MLP StopWords removed': 0.11590502326691325,
 'rorus SVM StopWords kept': 0.16618541940102613,
 'rorus MultinomialNB StopWords kept': 0.11721751580956927,
 'rorus MLP StopWords kept': 0.15065028039613412,
 '(tf-idf) rorus SVM StopWords removed': 0.1185777353537764,
 '(tf-idf) rorus MultinomialNB StopWords removed': 0.11013005607922682,
 '(tf-idf) rorus MLP StopWords removed': 0.10686075647297459,
 '(tf-idf) rorus SVM StopWords kept': 0.16098317623195324,
 '(tf-idf) rorus MultinomialNB StopWords kept': 0.11633456628087341,
 '(tf-idf) rorus MLP StopWords kept': 0.1378833074812075,
 'rosrus HMM StopWords removed': 0.09540627610070397,
 'rosrus HMM StopWords kept': 0.02586803484071113,
 '(tf-idf) rosrus HMM StopWords removed': 0.04827586206896552,
 '(tf-idf) rosrus HMM StopWords kept': 0.06199737501491469}

In [160]:
results_POS_rosrus=joblib.load('results_POS_rosrus')
results_POS_rosrus

{'rorus SVM StopWords removed': 0.1299128982221692,
 'rorus MultinomialNB StopWords removed': 0.10485622240782723,
 'rorus MLP StopWords removed': 0.11590502326691325,
 'rorus SVM StopWords kept': 0.16618541940102613,
 'rorus MultinomialNB StopWords kept': 0.11721751580956927,
 'rorus MLP StopWords kept': 0.15065028039613412,
 '(tf-idf) rorus SVM StopWords removed': 0.1185777353537764,
 '(tf-idf) rorus MultinomialNB StopWords removed': 0.11013005607922682,
 '(tf-idf) rorus MLP StopWords removed': 0.10686075647297459,
 '(tf-idf) rorus SVM StopWords kept': 0.16098317623195324,
 '(tf-idf) rorus MultinomialNB StopWords kept': 0.11633456628087341,
 '(tf-idf) rorus MLP StopWords kept': 0.1378833074812075,
 'rosrus HMM StopWords removed': 0.024865767808137453,
 'rosrus HMM StopWords kept': 0.02586803484071113}

In [202]:
with open("results_POS_rosrus.txt", "w", encoding="utf-8") as f:
    for key, value in results_POS_rosrus.items():
        f.write(f"{key}: {value}\n")